# 09 OSM Arrival Cost

Replaces the heuristic `arrival_cost_m` with real pedestrian network distances
from OpenStreetMap, using **OSRM** (Open Source Routing Machine).

**Why OSRM instead of Overpass/osmnx?**  
OSRM has all OSM roads pre-computed. You send it two coordinates and get back
the actual walk distance in milliseconds — no graph downloads, no Overpass API,
no timeouts. It's free and needs no API key.

**Run time:** ~10–15 minutes for 393 places (~1 second each).  
**Re-runs:** instant — completed rows are cached and skipped.

Run all cells top to bottom. A progress bar will show estimated time remaining.

## Setup

In [10]:
from pathlib import Path
import sys, time

import pandas as pd
import numpy as np
import requests
from tqdm.notebook import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.metrics import haversine_meters

PROCESSED  = PROJECT_ROOT / "data" / "processed"
GT_PATH    = PROCESSED / "ground_truth_combined.csv"
CACHE_PATH = PROCESSED / "osm_routing_cache.csv"
OUTPUT_PATH  = PROCESSED / "osm_arrival_cost.csv"
SUMMARY_PATH = PROCESSED / "osm_arrival_cost_summary.txt"

df = pd.read_csv(GT_PATH)
print(f"Loaded {len(df)} rows")


Loaded 3425 rows


## Quick connectivity test

Verifies OSRM is reachable before starting the full run.

In [11]:
OSRM_BASE   = "http://router.project-osrm.org/route/v1/foot"
TIMEOUT     = 10   # seconds per request
DELAY       = 1.0  # seconds between requests — polite to the free server
MAX_RETRIES = 3

# Quick test — SF Ferry Building to a nearby point
test = requests.get(
    f"{OSRM_BASE}/-122.3941,37.7955;-122.3944,37.7958",
    params={"overview": "false"}, timeout=TIMEOUT
)
data = test.json()
print(f"OSRM status: {data['code']}")
print(f"Test route:  {data['routes'][0]['distance']:.1f}m walk  "
      f"({data['routes'][0]['duration']:.0f}s)")
print("Ready to route.")


OSRM status: Ok
Test route:  42.5m walk  (4s)
Ready to route.


## Routing function

In [12]:
def route_one(row, max_retries=MAX_RETRIES):
    """
    Compute pedestrian walk distance via OSRM.
    Returns dict with walk_m, routing_penalty_m, routing_method.
    Retries up to max_retries times with backoff.
    """
    place_id = str(row.get("id", row.name))
    cur_lat, cur_lon = row["current_lat"], row["current_lon"]
    gt_lat,  gt_lon  = row["gt_lat"],      row["gt_lon"]
    straight = round(haversine_meters(cur_lat, cur_lon, gt_lat, gt_lon), 2)

    url = f"{OSRM_BASE}/{cur_lon},{cur_lat};{gt_lon},{gt_lat}"

    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params={"overview": "false"}, timeout=TIMEOUT)
            resp.raise_for_status()
            data = resp.json()

            if data.get("code") != "Ok" or not data.get("routes"):
                return {"id": place_id, "walk_m": straight,
                        "straight_line_m": straight, "routing_penalty_m": 0.0,
                        "routing_method": f"fallback (code={data.get('code')})"}

            walk_m  = round(data["routes"][0]["distance"], 2)
            penalty = round(max(0.0, walk_m - straight), 2)
            return {"id": place_id, "walk_m": walk_m,
                    "straight_line_m": straight, "routing_penalty_m": penalty,
                    "routing_method": "osrm"}

        except Exception as e:
            if attempt < max_retries:
                time.sleep(5 * attempt)   # 5s, 10s backoff
            else:
                return {"id": place_id, "walk_m": straight,
                        "straight_line_m": straight, "routing_penalty_m": 0.0,
                        "routing_method": f"fallback ({type(e).__name__})"}

print("route_one() defined")


route_one() defined


## Load cache and identify rows to compute

In [13]:
# Load cache — only keep successful rows, ignore old failures
if CACHE_PATH.exists():
    cache_df = pd.read_csv(CACHE_PATH)
    good = cache_df[cache_df["routing_method"].isin(["osrm", "osmnx", "osmnx_same_node"])]
    cache = {str(r["id"]): r.to_dict() for _, r in good.iterrows()}
    print(f"Cache loaded: {len(cache)} successful rows")
else:
    cache = {}
    print("No cache — starting fresh")

needs_routing = df[df["offset_haversine_m"] > 0].copy()
already_done  = {str(i) for i in needs_routing["id"].astype(str) if str(i) in cache}
to_compute    = needs_routing[~needs_routing["id"].astype(str).isin(already_done)]

print(f"\nTotal rows:          {len(df)}")
print(f"Zero-offset (skip):  {(df['offset_haversine_m'] == 0).sum()}")
print(f"Needs routing:       {len(needs_routing)}")
print(f"Already cached:      {len(already_done)}")
print(f"To compute now:      {len(to_compute)}")


Cache loaded: 393 successful rows

Total rows:          3425
Zero-offset (skip):  3025
Needs routing:       393
Already cached:      393
To compute now:      0


## Run routing

One request per place, 1-second pause between each. Progress bar shows ETA.  
**Safe to interrupt and re-run** — cache saves every 10 rows and already-done rows are skipped.

In [14]:
rows_list = list(to_compute.iterrows())

for i, (_, row) in enumerate(tqdm(rows_list, desc="OSRM routing")):
    place_id = str(row["id"])
    if place_id in cache:
        continue

    result = route_one(row)
    cache[place_id] = result

    if (i + 1) % 10 == 0 or (i + 1) == len(rows_list):
        pd.DataFrame(list(cache.values())).to_csv(CACHE_PATH, index=False)

    time.sleep(DELAY)

pd.DataFrame(list(cache.values())).to_csv(CACHE_PATH, index=False)

methods = pd.DataFrame(list(cache.values()))["routing_method"].value_counts()
print(f"\nDone. {len(cache)} rows cached.")
print(methods.to_string())


OSRM routing: 0it [00:00, ?it/s]


Done. 393 rows cached.
routing_method
osrm               365
osmnx               16
osmnx_same_node     12


## Assemble output and compare to heuristic

In [15]:
routing_df = pd.DataFrame(list(cache.values()))
routing_df["id"] = routing_df["id"].astype(str)

zero_ids  = df[df["offset_haversine_m"] == 0]["id"].astype(str)
zero_rows = pd.DataFrame({
    "id": zero_ids, "walk_m": 0.0, "straight_line_m": 0.0,
    "routing_penalty_m": 0.0, "routing_method": "zero_offset",
})

all_routing = pd.concat([routing_df, zero_rows], ignore_index=True)
all_routing["id"] = all_routing["id"].astype(str)

result = df.copy()
result["id"] = result["id"].astype(str)
result = result.merge(
    all_routing[["id", "walk_m", "routing_penalty_m", "routing_method"]],
    on="id", how="left",
)
result["osm_arrival_cost_m"] = result["walk_m"].fillna(result["offset_haversine_m"])
result.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(result)} rows → {OUTPUT_PATH}")


Saved 3425 rows → /Users/shivanibelambe/Pin-To-Place/data/processed/osm_arrival_cost.csv


## Results

In [16]:
old = result["offset_haversine_m"]
new = result["osm_arrival_cost_m"]
penalty = result["routing_penalty_m"].fillna(0)

comparison = pd.DataFrame({
    "metric":        ["mean", "median", "p90", "p95", "max"],
    "haversine_m":   [old.mean(), old.median(), old.quantile(.9), old.quantile(.95), old.max()],
    "osrm_m":        [new.mean(), new.median(), new.quantile(.9), new.quantile(.95), new.max()],
}).round(2)
comparison["diff_m"] = (comparison["osrm_m"] - comparison["haversine_m"]).round(2)
print("Haversine vs OSRM arrival cost:")
print(comparison.to_string(index=False))


Haversine vs OSRM arrival cost:
metric  haversine_m  osrm_m  diff_m
  mean         2.69   11.18    8.49
median         0.00    0.00    0.00
   p90        20.01    6.86  -13.15
   p95        23.34   25.70    2.36
   max        74.77 6478.50 6403.73


In [17]:
print("Routing penalty (extra walking due to real-world obstacles):")
print(penalty[penalty > 0].describe().round(2).to_string())
print()
print(f"Places with any routing penalty:  {(penalty > 0).sum()}")
print(f"Penalty >= 10m:                   {(penalty >= 10).sum()}")
print(f"Penalty >= 25m:                   {(penalty >= 25).sum()}")
print()

top = (
    result[penalty >= 10]
    .sort_values("routing_penalty_m", ascending=False)
    [["name","category_primary","tier_label",
      "offset_haversine_m","walk_m","routing_penalty_m"]]
    .head(15)
)
top


Routing penalty (extra walking due to real-world obstacles):
count     183.00
mean      169.47
std       523.07
min         0.03
25%        15.26
50%        44.58
75%       140.61
max      6451.99

Places with any routing penalty:  183
Penalty >= 10m:                   150
Penalty >= 25m:                   113



,name,category_primary,tier_label,offset_haversine_m,walk_m,routing_penalty_m
2278,Life Storage,self_storage_facility,standard_commercial,26.511476,6478.5,6451.99
2276,Raising Cane's,chicken_wings_restaurant,standard_commercial,25.593140,1877.2,1851.61
1451,Rincon Alegre,bar,standard_commercial,26.553433,1371.0,1344.45
3365,Grand Reserve at Canton,accommodation,standard_commercial,28.025943,1073.4,1045.37
742,Olive Garden Italian Restaurant,salad_bar,standard_commercial,22.529941,859.1,836.57
1279,Olive Garden Italian Restaurant,salad_bar,standard_commercial,23.444465,759.2,735.76
2303,Sun Resorts & Residences Ft. Myers Beach,campground,open_space,23.490443,727.3,703.81
1171,La Quinta Inn & Suites by Wyndham Wytheville,resort,standard_commercial,24.362991,652.5,628.14
3260,Divine Audio Visual,home_theater_systems_stores,standard_commercial,22.177090,642.1,619.92
2113,Ahnala,american_restaurant,standard_commercial,28.251072,639.4,611.15


In [18]:
print("Mean routing penalty by tier:")
print(result.groupby("tier_label")["routing_penalty_m"].agg(
    count="count",
    mean_penalty="mean",
    pct_any_penalty=lambda x: (x > 0).mean() * 100,
).round(2).to_string())

# Save summary
from src.osm_routing import write_summary
write_summary(result)
print(f"\nSummary saved → {SUMMARY_PATH}")


Mean routing penalty by tier:
                     count  mean_penalty  pct_any_penalty
tier_label                                               
multi_tenant           163          1.24             1.23
no_building            746          0.00             0.00
open_space             209          8.91            12.44
standard_commercial   2300         12.59             6.72

Summary saved → /Users/shivanibelambe/Pin-To-Place/data/processed/osm_arrival_cost_summary.txt
